# CMPE 249 detection: Stage 2 runner
Runtime → Change runtime type → **G4 GPU** (use the same GPU type for every latency number).

No Google Drive: data lives on Colab local disk, and results are pushed to this repo's `results` branch after every run.
One-time setup: add a GitHub token as a Colab Secret named `GITHUB_TOKEN` (key icon in the left sidebar), with
*Contents: read and write* on `shloakk/2d-object-detection` only.

Every cell is safe to re-run: clones are pulled instead of re-cloned, and data already on the
runtime disk is not downloaded again.

In [1]:
# Which runtime is this? (GPU model, TPU or CPU)
import glob, os, shutil, subprocess
SHORT_NAMES = {'L4': 'L4', 'A100': 'A100', 'H100': 'H100', 'T4': 'T4', 'RTX PRO 6000': 'G4'}
gpu = ''
if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip()
if gpu:
    short = next((v for k, v in SHORT_NAMES.items() if k in gpu), 'other GPU')
    print(f'Runtime: {short} GPU  ({gpu})')
elif glob.glob('/dev/accel*') or glob.glob('/dev/vfio/[0-9]*') or os.environ.get('TPU_ACCELERATOR_TYPE'):
    print(f"Runtime: TPU ({os.environ.get('TPU_ACCELERATOR_TYPE', 'type not reported')}): these PyTorch models need a GPU")
else:
    print('Runtime: CPU only: switch to a GPU runtime')
if gpu and short != 'G4':
    print('Note: every Stage 2 latency number should come from the same GPU type (G4 planned).')

Runtime: G4 GPU  (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB)


In [8]:
import os
REPO = '/content/2d-object-detection'
if os.path.isdir(f'{REPO}/.git'):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/shloakk/2d-object-detection.git {REPO}
%cd {REPO}
!git log -1 --oneline
!pip install -q -r requirements.txt

/content/2d-object-detection
3e4cbe0 (HEAD -> main, origin/main, origin/HEAD) Stage 3a: KITTI fine-tune, TensorRT FP16 export, fine-tune figure and deploy table


In [3]:
import os
# Results go into a clone of the `results` branch; src.sync_results commits and pushes it.
if os.path.isdir('/content/results/.git'):
    !git -C /content/results pull -q --rebase
else:
    from getpass import getpass
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = getpass('GitHub token: ')
    !git clone -q -b results https://{token}@github.com/shloakk/2d-object-detection.git /content/results
    !git -C /content/results config user.name 'Colab runner'
    !git -C /content/results config user.email 'colab@users.noreply.github.com'
os.environ['CMPE249_DATA'] = '/content/data'        # Colab local disk: fast reads
os.environ['CMPE249_WEIGHTS'] = '/content/weights'
os.environ['CMPE249_RESULTS'] = '/content/results'
import torch; print(torch.__version__, torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no CUDA GPU')

2.15.0+cu132 NVIDIA RTX PRO 6000 Blackwell Server Edition


In [4]:
# Data prep. Safe to re-run: each script checks what is already on the runtime disk and only
# downloads what is missing (the disk is wiped whenever the runtime resets).
!python -m src.data.kitti_to_coco            # full KITTI val (1496 images)
!python -m src.data.kitti_scene_split        # Stage 3a scene-disjoint split (whole drives held out)
!python -m src.data.coco_subset --full       # full COCO val2017 (5000 images, fetched by coco_url)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Unzipping /content/data/kitti.zip to /content/data/kitti...: 100% ━━━━━━━━━━━━ 14969/14969 12.0Kfiles/s 1.2s0.1s
kitti.yaml class order: {0: 'car', 1: 'van', 2: 'truck', 3: 'pedestrian', 4: 'person_sitting', 5: 'cyclist', 6: 'tram', 7: 'misc'}

1496 images from /content/data/kitti/images/val
Instances before mapping (KITTI classes):
  car               5716
  van                555
  truck              223
  pedestrian         921
  person_sitting      36
  cyclist            352
  tram               117
  misc               208
Instances after mapping (COCO ids):
  car -> car                     5716
  cyclist -> ignore person        352
  pedestrian -> person            921
  person_sitting -> ig

In [5]:
# One config end to end, then push its results to the `results` branch
!python -m src.predict --config configs/full/yolo26s_kitti.yaml && python -m src.sync_results

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
[yolo26s_kitti_full_20261031-004359] yolo26s on kitti (1496 images) device=cuda

284823 detections -> /content/results/yolo26s_kitti_full_20261031-004359/detections.json
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=4.22s).
Accumulating evaluation results...
DONE (t=0.62s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.256
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.494
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.235
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.182
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.225
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.389
 Average 

In [6]:
# All accuracy runs (Faster R-CNN COCO is the slowest). Each run is pushed as soon as it finishes.
for cfg in ['yolo26s_coco', 'frcnn_v2_coco', 'rtdetr_l_coco', 'yolo26s_o2m_coco',
            'rtdetr_l_kitti', 'frcnn_v2_kitti', 'yolo26s_o2m_kitti', 'yolo26s_1280_kitti']:
    !python -m src.predict --config configs/full/{cfg}.yaml && python -m src.sync_results

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
[yolo26s_coco_full_20261031-004426] yolo26s on coco (5000 images) device=cuda

641318 detections -> /content/results/yolo26s_coco_full_20261031-004426/detections.json
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=17.03s).
Accumulating evaluation results...
DONE (t=3.26s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.474
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.638
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.517
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.295
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.520
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.640
 Average Re

In [7]:
# Latency: batch 1, 50 warm-up + 200 timed, FP32 and FP16, on 200 KITTI images
for m in ['yolo26s', 'yolo26s_o2m', 'yolo26s_1280', 'rtdetr_l', 'frcnn_v2']:
    for half in ['', '--half']:
        !python -m src.bench_latency --config configs/full/{m}_kitti.yaml {half}
!python -m src.sync_results

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
[yolo26s_kitti_full_latency_fp32_conf0.001_20261031-005328] yolo26s fp32 on cuda: 50 warm-up + 200 timed runs
pre 0.35  infer 3.00  post 0.16 ms | e2e p50 3.67  p95 3.69 ms | peak mem 77.28515625 -> /content/results/yolo26s_kitti_full_latency_fp32_conf0.001_20261031-005328
/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (
[yolo26s_kitti_full_latency_fp16_conf0.001_20261031-005332] yolo26s fp16 on cuda: 50 warm-up + 200 timed runs
pre 0.35  infer 3.26  post 0.17 ms | e2e p50 3.95  p95 3.98 ms | peak mem 54.35791015625 -> /content/results/yolo26s_kitti_full_latency_fp16_conf0.001_20261031-005332
/usr/local/lib/python3.13/dis

## Stage 3a: KITTI fine-tune + TensorRT FP16
Needs cells 1–4 first (repo, results clone, data incl. the scene-disjoint split).

Ultralytics' kitti.yaml splits frames at random, so val scenes also appear in train (the first fine-tune
scored 72.0 mAP on it, mostly memorised scenes). The fine-tune here uses the scene-disjoint split instead:
20 whole drives held out as val (1,559 frames), 5,922 train frames. The zero-shot YOLO26s is scored on the
same scene val. The fine-tune keeps `last.pt` and pushes it with the results; after a runtime reset, run
the restore cell instead of training again.

In [ ]:
# Fine-tune YOLO26s on scene-disjoint KITTI train: 50 epochs, imgsz 640, seed 0, images cached in RAM
!python -m src.finetune --split scene --epochs 50 && python -m src.sync_results

In [ ]:
# Only after a runtime reset: restore the scene fine-tune weights from the results branch
import glob, os, shutil
runs = sorted(glob.glob('/content/results/yolo26s_kitti_scene_finetune_e50_*/last.pt'))
os.makedirs('/content/weights', exist_ok=True)
shutil.copy2(runs[-1], '/content/weights/yolo26s_kitti_scene_ft.pt'); print('restored', runs[-1])

In [ ]:
# Accuracy on the scene-disjoint val: zero-shot, fine-tuned (rect 640), fine-tuned square 640 (pairs with TensorRT)
for cfg in ['yolo26s_scene_kitti', 'yolo26s_ft_scene_kitti', 'yolo26s_ft_scene_sq_kitti']:
    !python -m src.predict --config configs/full/{cfg}.yaml && python -m src.sync_results

In [ ]:
# TensorRT FP16 engine (static 1x3x640x640, NMS-free head), then its accuracy and latency.
# TensorRT 11 FP16 export needs nvidia-modelopt; Ultralytics' auto-install pins torch==2.15.0+cu132,
# which is not on PyPI, so install it here against the torch already present.
!pip install -q "nvidia-modelopt[onnx]>=0.44" && python -c "import modelopt, torch; print('modelopt', modelopt.__version__, 'torch', torch.__version__)"
!python -m src.export_trt && python -m src.sync_results
!python -m src.predict --config configs/full/yolo26s_ft_scene_trt_kitti.yaml && python -m src.sync_results
!python -m src.bench_latency --config configs/full/yolo26s_ft_scene_trt_kitti.yaml
# PyTorch references on the same square input, FP32 and FP16
!python -m src.bench_latency --config configs/full/yolo26s_ft_scene_sq_kitti.yaml
!python -m src.bench_latency --config configs/full/yolo26s_ft_scene_sq_kitti.yaml --half
!python -m src.sync_results